<a href="https://colab.research.google.com/github/rafa1485/Metodologias-Innovadoras-Pronostico-Series-Nominales/blob/main/ajuste_y_comparacion_modelos_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

INPUT_EXCEL = "/content/Input_series.xlsx"

OUTPUT_SAMPLES_EXCEL = "muestras_entrenamiento.xlsx"
OUTPUT_TARGETS_EXCEL = "outputs_entrenamiento.xlsx"
OUTPUT_SKIPPED_EXCEL = "muestras_descartadas.xlsx"

# Los nombres de columnas están en la fila 9 de Excel.
# pandas cuenta desde 0, entonces header=8.
EXCEL_HEADER_ROW = 8

WINDOW_SIZE = 24

RANDOM_STEPS = [2, 3]

RANDOM_SEED = 20

# Nombre de la variable que se desea predecir.
# Debe coincidir exactamente con el nombre de una columna del Excel.
TARGET_COLUMN = "vsup"

# Lista o diccionario con los periodos faltantes finales por variable.
#
# Opción 1: lista, alineada con las columnas de series, sin contar la columna periodo.
# Si el Excel tiene:
#   Periodo | U1 | U2 | U3 | U4
# entonces:
#   MISSING_PERIODS = [0, 1, 2, 3]
#
# Opción 2: diccionario por nombre de variable.
#   MISSING_PERIODS = {
#       "U1": 0,
#       "U2": 1,
#       "U3": 2,
#       "U4": 3,
#   }
#


MISSING_PERIODS = {
    "vsup": 0, #2,
    "vasmay": 0, #2,
    "vcc": 0, #2,
    "velec": 0, #2,
    "vmcame": 0,
    "argiva": 0, #1,
    "ica_arg": 0,
    "bc_arg": 0,
}

# Política frente a faltantes iniciales dentro de una ventana.
#
# "skip_sample":
#     Descarta la muestra si alguna columna conserva NaN luego de imputar
#     los faltantes finales con SARIMAX.
#
# "leave_blank":
#     Deja los NaN en el Excel. No recomendado si luego se entrena una red
#     directamente con esas muestras.
#
# "fill_first_valid":
#     Rellena los NaN iniciales con el primer valor observado de esa columna
#     dentro de la muestra. Es una imputación fuerte y debe usarse con cuidado.
#
INITIAL_MISSING_POLICY = "skip_sample"

# Configuración SARIMAX.
# Para series mensuales con estacionalidad anual:
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEASONAL_ORDER = (1, 1, 1, 12)

# Si no se quiere estacionalidad:
# SARIMAX_SEASONAL_ORDER = (0, 0, 0, 0)

MIN_HISTORY_FOR_SARIMAX = 18


# ============================================================
# LECTURA DEL EXCEL
# ============================================================

def read_time_series_excel(path: str, header_row: int = 8) -> pd.DataFrame:
    """
    Lee el Excel usando como encabezado la fila 9.
    La primera columna es la etiqueta del periodo.
    Todas las demás columnas son series temporales.
    """

    df = pd.read_excel(path, header=header_row)

    df = df.dropna(how="all").reset_index(drop=True)

    if df.shape[1] < 2:
        raise ValueError(
            "El archivo debe tener al menos una columna de periodo y una columna de serie."
        )

    # Quitar columnas completamente vacías, por si el Excel tiene columnas residuales.
    df = df.dropna(axis=1, how="all")

    period_col = df.columns[0]
    series_cols = list(df.columns[1:])

    # Convertir las columnas de series a numéricas.
    for col in series_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print("Archivo leído correctamente.")
    print(f"Columna de periodo: {period_col}")
    print(f"Columnas de series: {series_cols}")

    return df


# ===============================================================
# FUNCION QUE IMPRIME EL TAMAÑO DEL DATASET
# ===============================================================

def print_dataset_size(samples: list[dict], targets_df: pd.DataFrame):
    """
    Imprime el tamaño del dataset generado.
    """

    n_samples = len(samples)

    if n_samples == 0:
        print("No se generaron muestras.")
        return

    first_sample_df = samples[0]["sample_df"]

    window_size = first_sample_df.shape[0]
    n_columns_total = first_sample_df.shape[1]

    # Restamos 1 porque la primera columna es la etiqueta del periodo
    n_series = n_columns_total - 1

    print()
    print("Tamaño del dataset generado:")
    print(f"  X: ({n_samples}, {window_size}, {n_series})")
    print(f"  y: ({len(targets_df)},)")
    print(f"  Cantidad de muestras: {n_samples}")
    print(f"  Periodos por muestra: {window_size}")
    print(f"  Series por muestra: {n_series}")

# ============================================================
# CONFIGURACIÓN DE FALTANTES FINALES
# ============================================================

def normalize_missing_periods(missing_periods, series_cols: list[str]) -> dict:
    """
    Convierte missing_periods a un diccionario:
        {nombre_columna: cantidad_faltantes_finales}
    """

    if isinstance(missing_periods, dict):
        result = {}

        for col in series_cols:
            result[col] = int(missing_periods.get(col, 0))

        unknown_cols = set(missing_periods.keys()) - set(series_cols)
        if unknown_cols:
            raise ValueError(
                f"Hay variables en MISSING_PERIODS que no existen en el Excel: {unknown_cols}"
            )

        return result

    if isinstance(missing_periods, list):
        if len(missing_periods) != len(series_cols):
            raise ValueError(
                f"MISSING_PERIODS debe tener {len(series_cols)} valores, "
                f"uno por cada serie: {series_cols}"
            )

        return {
            col: int(k)
            for col, k in zip(series_cols, missing_periods)
        }

    raise TypeError("MISSING_PERIODS debe ser una lista o un diccionario.")


# ============================================================
# SARIMAX PARA IMPUTACIÓN
# ============================================================

def fit_sarimax_and_forecast(
    serie: pd.Series,
    steps: int,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    min_history: int = 18,
) -> np.ndarray:
    """
    Ajusta SARIMAX sobre la historia disponible y pronostica 'steps' valores.

    Si hay muy pocos datos o el modelo falla, usa como respaldo el último
    valor observado.
    """

    serie = pd.to_numeric(serie, errors="coerce")

    # Se eliminan faltantes iniciales y cualquier otro NaN histórico.
    serie = serie.dropna()

    if steps <= 0:
        return np.array([])

    if len(serie) == 0:
        return np.repeat(np.nan, steps)

    if len(serie) < min_history:
        last_value = serie.iloc[-1]
        return np.repeat(last_value, steps)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            model = SARIMAX(
                serie,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )

            result = model.fit(disp=False)
            forecast = result.forecast(steps=steps)

        return forecast.to_numpy()

    except Exception:
        last_value = serie.iloc[-1]
        return np.repeat(last_value, steps)


# ============================================================
# MANEJO DE FALTANTES INICIALES
# ============================================================

def handle_initial_missing_values(
    sample: pd.DataFrame,
    series_cols: list[str],
    policy: str,
) -> tuple[pd.DataFrame, bool, str]:
    """
    Maneja faltantes que permanecen dentro de la ventana luego de imputar
    los faltantes finales.

    Devuelve:
        sample, is_valid, reason
    """

    remaining_nan_cols = [
        col for col in series_cols
        if sample[col].isna().any()
    ]

    if not remaining_nan_cols:
        return sample, True, ""

    if policy == "skip_sample":
        reason = (
            "La muestra conserva faltantes iniciales o internos en las columnas: "
            + ", ".join(remaining_nan_cols)
        )
        return sample, False, reason

    if policy == "leave_blank":
        return sample, True, ""

    if policy == "fill_first_valid":
        sample = sample.copy()

        for col in remaining_nan_cols:
            first_valid = sample[col].dropna()

            if len(first_valid) == 0:
                reason = f"La columna {col} no tiene ningún valor válido dentro de la muestra."
                return sample, False, reason

            sample[col] = sample[col].fillna(first_valid.iloc[0])

        return sample, True, ""

    raise ValueError(
        "INITIAL_MISSING_POLICY debe ser: "
        "'skip_sample', 'leave_blank' o 'fill_first_valid'."
    )


# ============================================================
# CONSTRUCCIÓN DE UNA MUESTRA
# ============================================================

def build_sample_with_imputation(
    df: pd.DataFrame,
    end_idx: int,
    window_size: int,
    missing_periods_by_col: dict,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    initial_missing_policy: str = "skip_sample",
) -> tuple[pd.DataFrame, bool, str]:
    """
    Construye una muestra de longitud window_size terminando en end_idx.

    Para cada columna:
    - toma los últimos window_size periodos;
    - simula que los últimos k periodos no estaban disponibles;
    - imputa esos k periodos con SARIMAX;
    - controla faltantes iniciales o internos.
    """

    period_col = df.columns[0]
    series_cols = list(df.columns[1:])

    start_idx = end_idx - window_size + 1

    if start_idx < 0:
        return pd.DataFrame(), False, "No hay suficientes periodos para construir la ventana."

    sample = df.iloc[start_idx:end_idx + 1].copy().reset_index(drop=True)

    for col_name in series_cols:
        k_missing = int(missing_periods_by_col.get(col_name, 0))

        if k_missing <= 0:
            continue

        if k_missing >= window_size:
            return (
                sample,
                False,
                f"La cantidad de faltantes finales para {col_name} "
                f"no puede ser mayor o igual que WINDOW_SIZE."
            )

        available_until_idx = end_idx - k_missing

        if available_until_idx < 0:
            return (
                sample,
                False,
                f"No hay historia suficiente para imputar {col_name}."
            )

        history = df.loc[:available_until_idx, col_name]

        forecast_values = fit_sarimax_and_forecast(
            serie=history,
            steps=k_missing,
            order=order,
            seasonal_order=seasonal_order,
            min_history=MIN_HISTORY_FOR_SARIMAX,
        )

        positions_to_impute = range(window_size - k_missing, window_size)

        for pos, imputed_value in zip(positions_to_impute, forecast_values):
            sample.loc[pos, col_name] = imputed_value

    sample, is_valid, reason = handle_initial_missing_values(
        sample=sample,
        series_cols=series_cols,
        policy=initial_missing_policy,
    )

    return sample, is_valid, reason


# ============================================================
# GENERACIÓN DE MUESTRAS DE ENTRENAMIENTO
# ============================================================

def generate_training_samples(
    df: pd.DataFrame,
    target_column: str,
    window_size: int,
    missing_periods_by_col: dict,
    random_steps: list[int],
    seed: int = 42,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    initial_missing_policy: str = "skip_sample",
):
    """
    Genera muestras de entrenamiento.

    Para una ventana que termina en end_idx:
        X = ventana [end_idx - window_size + 1, ..., end_idx]
        y = target_column en end_idx + 1

    Las ventanas se recorren hacia atrás con saltos aleatorios de 2 o 3 periodos.
    """

    random.seed(seed)

    period_col = df.columns[0]
    series_cols = list(df.columns[1:])

    if target_column not in series_cols:
        raise ValueError(
            f"La variable objetivo '{target_column}' no existe entre las columnas de series: "
            f"{series_cols}"
        )

    samples = []
    targets = []
    skipped = []

    # Debe existir el valor objetivo en end_idx + 1.
    end_idx = len(df) - 2

    sample_number = 1
    candidate_number = 1

    while end_idx - window_size + 1 >= 0:

        target_idx = end_idx + 1
        target_value = df.loc[target_idx, target_column]
        target_period = df.loc[target_idx, period_col]

        if pd.isna(target_value):
            skipped.append(
                {
                    "candidate_number": candidate_number,
                    "end_idx": end_idx,
                    "target_idx": target_idx,
                    "reason": f"Objetivo {target_column} en el {target_period} está vacío.",
                }
            )

            #end_idx -= random.choice(random_steps)
            end_idx -= 1
            candidate_number += 1
            continue

        sample_df, is_valid, reason = build_sample_with_imputation(
            df=df,
            end_idx=end_idx,
            window_size=window_size,
            missing_periods_by_col=missing_periods_by_col,
            order=order,
            seasonal_order=seasonal_order,
            initial_missing_policy=initial_missing_policy,
        )

        if not is_valid:
            skipped.append(
                {
                    "candidate_number": candidate_number,
                    "end_idx": end_idx,
                    "target_idx": target_idx,
                    "reason": reason,
                }
            )

            #end_idx -= random.choice(random_steps)
            end_idx -= 1
            candidate_number += 1
            continue

        samples.append(
            {
                "sample_number": sample_number,
                "candidate_number": candidate_number,
                "end_idx": end_idx,
                "target_idx": target_idx,
                "sample_df": sample_df,
            }
        )

        targets.append(
            {
                "sample_number": sample_number,
                "candidate_number": candidate_number,
                "input_start_period": sample_df.iloc[0][period_col],
                "input_end_period": sample_df.iloc[-1][period_col],
                "target_period": target_period,
                "target_variable": target_column,
                "target_value": target_value,
            }
        )

        step = random.choice(random_steps)
        end_idx -= step
        sample_number += 1
        candidate_number += 1

    targets_df = pd.DataFrame(targets)
    skipped_df = pd.DataFrame(skipped)

    print(f"end_idx: {end_idx}")
    print(f"len(df): {len(df)}")
    print("Dataframe")
    print(targets_df)
    print()
    print("Skipped")
    print(skipped_df)
    print("=================================")
    return samples, targets_df, skipped_df


# ============================================================
# GUARDADO EN EXCEL
# ============================================================

def save_samples_to_excel(samples: list[dict], output_path: str):
    """
    Guarda cada muestra en una hoja separada.
    """

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sample in samples:
            sample_number = sample["sample_number"]
            sample_df = sample["sample_df"]

            sheet_name = f"muestra_{sample_number:04d}"[:31]

            sample_df.to_excel(writer, sheet_name=sheet_name, index=False)


def save_targets_to_excel(targets_df: pd.DataFrame, output_path: str):
    """
    Guarda los outputs de entrenamiento.
    """

    targets_df.to_excel(output_path, index=False)


def save_skipped_to_excel(skipped_df: pd.DataFrame, output_path: str):
    """
    Guarda un registro de las muestras descartadas.
    """

    if len(skipped_df) > 0:
        skipped_df.to_excel(output_path, index=False)


# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

def main():
    input_path = Path(INPUT_EXCEL)

    if not input_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {INPUT_EXCEL}")

    df = read_time_series_excel(
        path=INPUT_EXCEL,
        header_row=EXCEL_HEADER_ROW,
    )

    period_col = df.columns[0]
    series_cols = list(df.columns[1:])

    missing_periods_by_col = normalize_missing_periods(
        missing_periods=MISSING_PERIODS,
        series_cols=series_cols,
    )

    print()
    print("Configuración de faltantes finales:")
    for col, k in missing_periods_by_col.items():
        print(f"  {col}: {k}")

    print()
    print(f"Variable objetivo: {TARGET_COLUMN}")
    print(f"Tamaño de ventana: {WINDOW_SIZE}")
    print(f"Política de faltantes iniciales: {INITIAL_MISSING_POLICY}")

    samples, targets_df, skipped_df = generate_training_samples(
        df=df,
        target_column=TARGET_COLUMN,
        window_size=WINDOW_SIZE,
        missing_periods_by_col=missing_periods_by_col,
        random_steps=RANDOM_STEPS,
        seed=RANDOM_SEED,
        order=SARIMAX_ORDER,
        seasonal_order=SARIMAX_SEASONAL_ORDER,
        initial_missing_policy=INITIAL_MISSING_POLICY,
    )

    print_dataset_size(samples, targets_df)
    save_samples_to_excel(samples, OUTPUT_SAMPLES_EXCEL)
    save_targets_to_excel(targets_df, OUTPUT_TARGETS_EXCEL)
    save_skipped_to_excel(skipped_df, OUTPUT_SKIPPED_EXCEL)

    print()
    print("Proceso finalizado.")
    print(f"Muestras generadas: {len(samples)}")
    print(f"Muestras descartadas: {len(skipped_df)}")
    print(f"Archivo de muestras: {OUTPUT_SAMPLES_EXCEL}")
    print(f"Archivo de outputs: {OUTPUT_TARGETS_EXCEL}")

    if len(skipped_df) > 0:
        print(f"Archivo de muestras descartadas: {OUTPUT_SKIPPED_EXCEL}")

In [41]:
main()

Archivo leído correctamente.
Columna de periodo: Código
Columnas de series: ['vsup', 'vasmay', 'vcc', 'velec', 'vmcame', 'argiva', 'ica_arg', 'bc_arg']

Configuración de faltantes finales:
  vsup: 0
  vasmay: 0
  vcc: 0
  velec: 0
  vmcame: 0
  argiva: 0
  ica_arg: 0
  bc_arg: 0

Variable objetivo: vsup
Tamaño de ventana: 24
Política de faltantes iniciales: skip_sample
end_idx: 22
len(df): 372
Dataframe
    sample_number  candidate_number  input_start_period  input_end_period  \
0               1                 9             2024.04           2026.03   
1               2                10             2024.02           2026.01   
2               3                11             2023.11           2025.10   
3               4                12             2023.09           2025.08   
4               5                13             2023.06           2025.05   
5               6                14             2023.04           2025.03   
6               7                15             2023.0

# Entrenamiento y ejecución de la comparación entre modelos

In [42]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from statsmodels.tsa.statespace.sarimax import SARIMAX

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

SAMPLES_EXCEL = "muestras_entrenamiento.xlsx"
TARGETS_EXCEL = "outputs_entrenamiento.xlsx"

OUTPUT_RESULTS_EXCEL = "comparacion_modelos_temporal_loo.xlsx"

WINDOW_SIZE = 24
TARGET_VALUE_COLUMN = "target_value"

ORDER_MODE = "auto"  # "auto", "target_period", "sample_number_desc"

MIN_TRAIN_SAMPLES = 12

RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# SELECCIÓN DE COLUMNAS
# ============================================================

# Lista de series que se usarán como entrada del modelo.
# Deben coincidir exactamente con los nombres de columnas de las hojas
# de muestras_entrenamiento.xlsx.
#
# Si SELECTED_SERIES = None, se usan todas las columnas de series disponibles.
#
# Ejemplo:
# SELECTED_SERIES = ["U1", "U2", "U3", "U5"]
#
SELECTED_SERIES = [
    "vsup", #
    #"vasmay",
    "vcc", # 6.6
    "velec", #
    "vmcame", #
    "argiva", #
    "ica_arg", #
    "bc_arg" #
    ] # None

# Si querés seleccionar manualmente:
# SELECTED_SERIES = [
#     "U1",
#     "U2",
#     "U3",
#     "U4",
# ]

# Variable objetivo. Si TARGET_VARIABLE_NAME = None, se intentará tomar
# desde la columna target_variable del archivo outputs_entrenamiento.xlsx.
TARGET_VARIABLE_NAME = None

# Ejemplo:
# TARGET_VARIABLE_NAME = "U1"


# ============================================================
# CONFIGURACIÓN SARIMA / SARIMAX
# ============================================================

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (0, 1, 1, 12)

SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEASONAL_ORDER = (0, 1, 1, 12)

# Si querés desactivar estacionalidad:
# SARIMA_SEASONAL_ORDER = (0, 0, 0, 0)
# SARIMAX_SEASONAL_ORDER = (0, 0, 0, 0)


# ============================================================
# CONFIGURACIÓN REDES NEURONALES PEQUEÑAS
# ============================================================

BATCH_SIZE = 4
EPOCHS = 250
PATIENCE = 35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3

CNN_CHANNELS = 8
CNN_KERNEL_SIZE = 3
CNN_DROPOUT = 0.30

TCN_CHANNELS = [8, 8]
TCN_KERNEL_SIZE = 2
TCN_DROPOUT = 0.30


# ============================================================
# UTILIDADES
# ============================================================

def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = math.sqrt(np.mean((y_true - y_pred) ** 2))

    mask = y_true != 0

    if np.any(mask):
        mape = np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100
    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
    }


def count_trainable_parameters(model):
    """
    Cuenta los parámetros entrenables de un modelo PyTorch.
    """
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def sheet_name_from_sample_number(sample_number: int) -> str:
    return f"muestra_{int(sample_number):04d}"


# ============================================================
# LECTURA DEL DATASET
# ============================================================

def infer_target_variable_from_metadata(targets_df: pd.DataFrame):
    """
    Determina la variable objetivo.

    Prioridad:
    1) TARGET_VARIABLE_NAME si fue definido manualmente.
    2) Columna target_variable en outputs_entrenamiento.xlsx.
    """

    if TARGET_VARIABLE_NAME is not None:
        return TARGET_VARIABLE_NAME

    if "target_variable" in targets_df.columns:
        unique_targets = targets_df["target_variable"].dropna().unique()

        if len(unique_targets) == 1:
            return unique_targets[0]

        if len(unique_targets) > 1:
            raise ValueError(
                "El archivo outputs_entrenamiento.xlsx contiene más de una "
                "variable objetivo en la columna 'target_variable'. "
                "Definí TARGET_VARIABLE_NAME manualmente."
            )

    raise ValueError(
        "No se pudo determinar la variable objetivo. "
        "Definí TARGET_VARIABLE_NAME o asegurate de que exista la columna "
        "'target_variable' en outputs_entrenamiento.xlsx."
    )


def load_dataset_from_generated_excels(
    samples_excel: str,
    targets_excel: str,
    selected_series=None,
):
    """
    Lee los Excel generados previamente.

    Devuelve:
        X: np.ndarray con forma (n_muestras, WINDOW_SIZE, n_features)
        y: np.ndarray con forma (n_muestras,)
        metadata_df: metadatos del archivo outputs_entrenamiento.xlsx
        feature_names: nombres de las columnas utilizadas como entrada
        target_variable: nombre de la variable objetivo
    """

    samples_path = Path(samples_excel)
    targets_path = Path(targets_excel)

    if not samples_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {samples_excel}")

    if not targets_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {targets_excel}")

    targets_df = pd.read_excel(targets_excel)

    required_columns = ["sample_number", TARGET_VALUE_COLUMN]

    for col in required_columns:
        if col not in targets_df.columns:
            raise ValueError(f"No se encontró la columna '{col}' en {targets_excel}")

    target_variable = infer_target_variable_from_metadata(targets_df)

    xls = pd.ExcelFile(samples_excel)
    existing_sheets = set(xls.sheet_names)

    X_list = []
    y_list = []
    metadata_rows = []
    feature_names = None

    for _, row in targets_df.iterrows():
        sample_number = int(row["sample_number"])
        sheet_name = sheet_name_from_sample_number(sample_number)

        if sheet_name not in existing_sheets:
            raise ValueError(
                f"No se encontró la hoja '{sheet_name}' en {samples_excel}"
            )

        sample_df = pd.read_excel(samples_excel, sheet_name=sheet_name)

        if sample_df.shape[0] != WINDOW_SIZE:
            raise ValueError(
                f"La hoja {sheet_name} tiene {sample_df.shape[0]} filas, "
                f"pero se esperaban {WINDOW_SIZE}."
            )

        all_series_cols = list(sample_df.columns[1:])

        if selected_series is None:
            current_feature_names = all_series_cols
        else:
            missing_selected = [
                col for col in selected_series
                if col not in all_series_cols
            ]

            if missing_selected:
                raise ValueError(
                    f"En la hoja {sheet_name} faltan columnas seleccionadas: "
                    + ", ".join(missing_selected)
                )

            current_feature_names = list(selected_series)

        if target_variable not in current_feature_names:
            raise ValueError(
                f"La variable objetivo '{target_variable}' no está incluida "
                f"en SELECTED_SERIES. Para SARIMA/SARIMAX y redes, la serie "
                f"objetivo debe formar parte de la entrada."
            )

        if feature_names is None:
            feature_names = current_feature_names
        elif current_feature_names != feature_names:
            raise ValueError(
                f"La hoja {sheet_name} tiene columnas distintas a las anteriores."
            )

        values = sample_df[current_feature_names].to_numpy(dtype=float)

        if np.isnan(values).any():
            raise ValueError(
                f"La hoja {sheet_name} contiene NaN en las columnas seleccionadas. "
                f"Revisar imputación antes de entrenar."
            )

        target_value = float(row[TARGET_VALUE_COLUMN])

        if np.isnan(target_value):
            raise ValueError(
                f"El output de la muestra {sample_number} contiene NaN."
            )

        X_list.append(values)
        y_list.append(target_value)
        metadata_rows.append(row.to_dict())

    X = np.stack(X_list, axis=0)
    y = np.asarray(y_list, dtype=float)
    metadata_df = pd.DataFrame(metadata_rows)

    return X, y, metadata_df, feature_names, target_variable


def order_dataset_temporally(X, y, metadata_df, order_mode="auto"):
    """
    Ordena las muestras de más antiguas a más recientes.

    Si puede interpretar target_period como fecha, ordena por fecha.
    Si no, usa sample_number descendente, asumiendo que muestra_0001
    es la más reciente.
    """

    metadata_df = metadata_df.copy()

    if order_mode in ["auto", "target_period"] and "target_period" in metadata_df.columns:
        parsed_dates = pd.to_datetime(metadata_df["target_period"], errors="coerce")

        if parsed_dates.notna().all():
            metadata_df["_sort_key"] = parsed_dates
            order_indices = metadata_df.sort_values("_sort_key").index.to_numpy()

            X_ord = X[order_indices]
            y_ord = y[order_indices]
            metadata_ord = (
                metadata_df
                .loc[order_indices]
                .drop(columns=["_sort_key"])
                .reset_index(drop=True)
            )

            return X_ord, y_ord, metadata_ord

        if order_mode == "target_period":
            raise ValueError(
                "No se pudo convertir target_period a fecha. "
                "Usar ORDER_MODE='sample_number_desc'."
            )

    if order_mode in ["auto", "sample_number_desc"]:
        order_indices = metadata_df.sort_values(
            "sample_number",
            ascending=False,
        ).index.to_numpy()

        X_ord = X[order_indices]
        y_ord = y[order_indices]
        metadata_ord = metadata_df.loc[order_indices].reset_index(drop=True)

        return X_ord, y_ord, metadata_ord

    raise ValueError(
        "ORDER_MODE debe ser 'auto', 'target_period' o 'sample_number_desc'."
    )


# ============================================================
# SARIMA / SARIMAX BASELINE
# ============================================================

def sarima_forecast_one_step(
    target_history,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 12),
):
    """
    SARIMA univariado.

    Devuelve:
        predicción, número de parámetros ajustados
    """

    target_history = pd.Series(target_history).dropna()

    if len(target_history) == 0:
        return np.nan, 0

    if len(target_history) < 8:
        return float(target_history.iloc[-1]), 0

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            model = SARIMAX(
                target_history,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )

            result = model.fit(disp=False)
            pred = result.forecast(steps=1)

        n_params = len(result.params)

        return float(pred.iloc[0]), n_params

    except Exception:
        return float(target_history.iloc[-1]), 0


def sarimax_forecast_one_step(
    target_window,
    exog_window,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 12),
):
    """
    SARIMAX con variables exógenas.

    Usa una alineación rezagada para evitar requerir exógenas futuras:

        exog(t-23) -> y(t-22)
        exog(t-22) -> y(t-21)
        ...
        exog(t-1)  -> y(t)
        exog(t)    -> predice y(t+1)

    Devuelve:
        predicción, número de parámetros ajustados
    """

    target_window = np.asarray(target_window, dtype=float).reshape(-1)
    exog_window = np.asarray(exog_window, dtype=float)

    if exog_window.ndim == 1:
        exog_window = exog_window.reshape(-1, 1)

    # Si llega transpuesta: (n_exog, window_size) -> (window_size, n_exog)
    if (
        exog_window.shape[0] != len(target_window)
        and exog_window.shape[1] == len(target_window)
    ):
        exog_window = exog_window.T

    # Recorte defensivo si hubiera alguna diferencia menor.
    if exog_window.shape[0] != len(target_window):
        min_len = min(len(target_window), exog_window.shape[0])
        target_window = target_window[-min_len:]
        exog_window = exog_window[-min_len:, :]

    if len(target_window) < 8:
        return float(target_window[-1]), 0

    if exog_window.shape[1] == 0:
        return float(target_window[-1]), 0

    endog_history = pd.Series(target_window[1:]).dropna()
    exog_history = exog_window[:-1, :]
    exog_next = exog_window[-1:, :]

    if len(endog_history) != exog_history.shape[0]:
        min_len = min(len(endog_history), exog_history.shape[0])
        endog_history = endog_history.iloc[-min_len:]
        exog_history = exog_history[-min_len:, :]

    if len(endog_history) < 8:
        return float(target_window[-1]), 0

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            model = SARIMAX(
                endog_history,
                exog=exog_history,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )

            result = model.fit(disp=False)

            pred = result.forecast(
                steps=1,
                exog=exog_next,
            )

        n_params = len(result.params)

        return float(pred.iloc[0]), n_params

    except Exception:
        return float(target_window[-1]), 0


# ============================================================
# DATASET PYTORCH
# ============================================================

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]

        # De:
        #   (window_size, n_features)
        # a:
        #   (n_features, window_size)
        x = x.transpose(0, 1)

        y = self.y[idx]

        return x, y


# ============================================================
# MODELO 1D-CNN PEQUEÑO
# ============================================================

class SmallCNN1D(nn.Module):
    def __init__(
        self,
        n_features,
        n_channels=8,
        kernel_size=3,
        dropout=0.30,
    ):
        super().__init__()

        padding = kernel_size - 1

        self.conv = nn.Conv1d(
            in_channels=n_features,
            out_channels=n_channels,
            kernel_size=kernel_size,
            padding=padding,
        )

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        self.regressor = nn.Sequential(
            nn.Linear(n_channels, 8),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        out = self.conv(x)

        # Quitar excedente del padding para conservar causalidad aproximada.
        out = out[:, :, :x.shape[-1]]

        out = self.relu(out)
        out = self.dropout(out)

        last_time_step = out[:, :, -1]

        y_hat = self.regressor(last_time_step)

        return y_hat.squeeze(-1)


# ============================================================
# MODELO TCN PEQUEÑO
# ============================================================

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x

        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation,
        dropout,
    ):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation,
        )

        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            padding=padding,
            dilation=dilation,
        )

        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        else:
            self.downsample = None

        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        if self.downsample is not None:
            residual = self.downsample(x)
        else:
            residual = x

        return self.final_relu(out + residual)


class SmallTCN(nn.Module):
    def __init__(
        self,
        n_features,
        channels,
        kernel_size=2,
        dropout=0.30,
    ):
        super().__init__()

        layers = []

        for i, out_channels in enumerate(channels):
            dilation = 2 ** i

            if i == 0:
                in_channels = n_features
            else:
                in_channels = channels[i - 1]

            layers.append(
                TemporalBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )

        self.network = nn.Sequential(*layers)
        self.regressor = nn.Linear(channels[-1], 1)

    def forward(self, x):
        out = self.network(x)
        last_time_step = out[:, :, -1]
        y_hat = self.regressor(last_time_step)

        return y_hat.squeeze(-1)


# ============================================================
# ENTRENAMIENTO DE REDES PARA UN FOLD
# ============================================================

def scale_fold(X_train, X_val, y_train):
    n_train, window_size, n_features = X_train.shape

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_2d = X_train.reshape(-1, n_features)
    X_val_2d = X_val.reshape(-1, n_features)

    X_train_scaled = scaler_X.fit_transform(X_train_2d)
    X_val_scaled = scaler_X.transform(X_val_2d)

    X_train_scaled = X_train_scaled.reshape(n_train, window_size, n_features)
    X_val_scaled = X_val_scaled.reshape(1, window_size, n_features)

    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

    return X_train_scaled, X_val_scaled, y_train_scaled, scaler_y


def train_neural_model_for_fold(
    model,
    X_train,
    y_train,
    X_val,
    device,
    batch_size,
    epochs,
    patience,
    learning_rate,
    weight_decay,
):
    train_dataset = TimeSeriesDataset(X_train, y_train)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    model = model.to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # (1, window_size, n_features) -> (1, n_features, window_size)
    X_val_tensor = X_val_tensor.transpose(1, 2).to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            pred = model(batch_X)
            loss = criterion(pred, batch_y)

            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        train_loss = float(np.mean(losses))

        if train_loss < best_loss:
            best_loss = train_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()

    with torch.no_grad():
        pred_scaled = model(X_val_tensor).cpu().numpy()[0]

    return float(pred_scaled)


# ============================================================
# LEAVE-ONE-OUT TEMPORAL
# ============================================================

def temporal_leave_one_out_comparison(
    X,
    y,
    metadata_df,
    feature_names,
    target_variable,
):
    n_samples, window_size, n_features = X.shape

    if n_samples <= MIN_TRAIN_SAMPLES:
        raise ValueError(
            f"No hay suficientes muestras. "
            f"n_samples={n_samples}, MIN_TRAIN_SAMPLES={MIN_TRAIN_SAMPLES}"
        )

    if target_variable not in feature_names:
        raise ValueError(
            f"La variable objetivo '{target_variable}' no está en feature_names."
        )

    target_col_idx = feature_names.index(target_variable)

    results = []

    for val_idx in range(MIN_TRAIN_SAMPLES, n_samples):
        print(f"Fold temporal: entrenar [0:{val_idx}] -> validar muestra {val_idx}")

        X_train = X[:val_idx]
        y_train = y[:val_idx]

        X_val = X[val_idx:val_idx + 1]
        y_val = y[val_idx]

        meta = metadata_df.iloc[val_idx].to_dict()

        # -----------------------------
        # 1) SARIMA baseline univariado
        # -----------------------------
        target_history = X_val[0, :, target_col_idx]

        pred_sarima, n_params_sarima = sarima_forecast_one_step(
            target_history=target_history,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
        )

        # -----------------------------
        # 2) SARIMAX baseline con exógenas
        # -----------------------------
        exog_indices = [
            idx for idx in range(n_features)
            if idx != target_col_idx
        ]

        if len(exog_indices) > 0:
            exog_window = X_val[0][:, exog_indices]

            exog_window = np.asarray(exog_window, dtype=float)

            if exog_window.ndim == 1:
                exog_window = exog_window.reshape(-1, 1)

            pred_sarimax, n_params_sarimax = sarimax_forecast_one_step(
                target_window=target_history,
                exog_window=exog_window,
                order=SARIMAX_ORDER,
                seasonal_order=SARIMAX_SEASONAL_ORDER,
            )
        else:
            pred_sarimax = pred_sarima
            n_params_sarimax = n_params_sarima

        # -----------------------------
        # Escalado para redes
        # -----------------------------
        X_train_scaled, X_val_scaled, y_train_scaled, scaler_y = scale_fold(
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
        )

        # -----------------------------
        # 3) 1D-CNN pequeña
        # -----------------------------
        set_seed(RANDOM_SEED + val_idx)

        cnn_model = SmallCNN1D(
            n_features=n_features,
            n_channels=CNN_CHANNELS,
            kernel_size=CNN_KERNEL_SIZE,
            dropout=CNN_DROPOUT,
        )

        n_params_cnn_1d = count_trainable_parameters(cnn_model)

        pred_cnn_scaled = train_neural_model_for_fold(
            model=cnn_model,
            X_train=X_train_scaled,
            y_train=y_train_scaled,
            X_val=X_val_scaled,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            patience=PATIENCE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        pred_cnn = scaler_y.inverse_transform(
            np.array([[pred_cnn_scaled]])
        )[0, 0]

        # -----------------------------
        # 4) TCN pequeña
        # -----------------------------
        set_seed(RANDOM_SEED + 10_000 + val_idx)

        tcn_model = SmallTCN(
            n_features=n_features,
            channels=TCN_CHANNELS,
            kernel_size=TCN_KERNEL_SIZE,
            dropout=TCN_DROPOUT,
        )

        n_params_tcn = count_trainable_parameters(tcn_model)

        pred_tcn_scaled = train_neural_model_for_fold(
            model=tcn_model,
            X_train=X_train_scaled,
            y_train=y_train_scaled,
            X_val=X_val_scaled,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            patience=PATIENCE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        pred_tcn = scaler_y.inverse_transform(
            np.array([[pred_tcn_scaled]])
        )[0, 0]

        row = {
            "fold": val_idx - MIN_TRAIN_SAMPLES + 1,
            "val_position_chronological": val_idx,
            "train_samples": val_idx,
            "y_real": y_val,

            "target_variable": target_variable,
            "selected_series": ", ".join(feature_names),
            "n_selected_series": len(feature_names),

            "pred_sarima": pred_sarima,
            "pred_sarimax": pred_sarimax,
            "pred_cnn_1d": pred_cnn,
            "pred_tcn": pred_tcn,

            "error_sarima": y_val - pred_sarima,
            "error_sarimax": y_val - pred_sarimax,
            "error_cnn_1d": y_val - pred_cnn,
            "error_tcn": y_val - pred_tcn,

            "abs_error_sarima": abs(y_val - pred_sarima),
            "abs_error_sarimax": abs(y_val - pred_sarimax),
            "abs_error_cnn_1d": abs(y_val - pred_cnn),
            "abs_error_tcn": abs(y_val - pred_tcn),

            "params_sarima": n_params_sarima,
            "params_sarimax": n_params_sarimax,
            "params_cnn_1d": n_params_cnn_1d,
            "params_tcn": n_params_tcn,
        }

        for key, value in meta.items():
            row[f"meta_{key}"] = value

        results.append(row)

    results_df = pd.DataFrame(results)

    return results_df


# ============================================================
# RESUMEN DE MÉTRICAS Y COMPLEJIDAD
# ============================================================

def build_metrics_summary(results_df):
    y_true = results_df["y_real"].to_numpy(dtype=float)

    model_columns = {
        "SARIMA": {
            "pred": "pred_sarima",
            "params": "params_sarima",
        },
        "SARIMAX": {
            "pred": "pred_sarimax",
            "params": "params_sarimax",
        },
        "1D-CNN pequeña": {
            "pred": "pred_cnn_1d",
            "params": "params_cnn_1d",
        },
        "TCN pequeña": {
            "pred": "pred_tcn",
            "params": "params_tcn",
        },
    }

    rows = []

    for model_name, cols in model_columns.items():
        y_pred = results_df[cols["pred"]].to_numpy(dtype=float)
        metrics = compute_metrics(y_true, y_pred)

        params_values = results_df[cols["params"]].to_numpy(dtype=float)

        rows.append(
            {
                "modelo": model_name,
                "MAE": metrics["MAE"],
                "RMSE": metrics["RMSE"],
                "MAPE": metrics["MAPE"],
                "parametros_promedio": np.nanmean(params_values),
                "parametros_min": np.nanmin(params_values),
                "parametros_max": np.nanmax(params_values),
                "n_selected_series": results_df["n_selected_series"].iloc[0],
                "target_variable": results_df["target_variable"].iloc[0],
                "selected_series": results_df["selected_series"].iloc[0],
            }
        )

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values("MAE").reset_index(drop=True)

    return summary_df


def build_model_complexity_summary(results_df):
    """
    Genera una tabla separada enfocada únicamente en complejidad.
    """

    rows = [
        {
            "modelo": "SARIMA",
            "parametros_promedio": results_df["params_sarima"].mean(),
            "parametros_min": results_df["params_sarima"].min(),
            "parametros_max": results_df["params_sarima"].max(),
        },
        {
            "modelo": "SARIMAX",
            "parametros_promedio": results_df["params_sarimax"].mean(),
            "parametros_min": results_df["params_sarimax"].min(),
            "parametros_max": results_df["params_sarimax"].max(),
        },
        {
            "modelo": "1D-CNN pequeña",
            "parametros_promedio": results_df["params_cnn_1d"].mean(),
            "parametros_min": results_df["params_cnn_1d"].min(),
            "parametros_max": results_df["params_cnn_1d"].max(),
        },
        {
            "modelo": "TCN pequeña",
            "parametros_promedio": results_df["params_tcn"].mean(),
            "parametros_min": results_df["params_tcn"].min(),
            "parametros_max": results_df["params_tcn"].max(),
        },
    ]

    return pd.DataFrame(rows)


# ============================================================
# PROGRAMA PRINCIPAL
# ============================================================

def modelos():
    set_seed(RANDOM_SEED)

    print(f"Dispositivo usado: {DEVICE}")

    X, y, metadata_df, feature_names, target_variable = load_dataset_from_generated_excels(
        samples_excel=SAMPLES_EXCEL,
        targets_excel=TARGETS_EXCEL,
        selected_series=SELECTED_SERIES,
    )

    X, y, metadata_df = order_dataset_temporally(
        X=X,
        y=y,
        metadata_df=metadata_df,
        order_mode=ORDER_MODE,
    )

    print()
    print("Dataset cargado y ordenado temporalmente.")
    print(f"X.shape: {X.shape}")
    print(f"y.shape: {y.shape}")
    print(f"Variable objetivo: {target_variable}")
    print(f"Variables seleccionadas: {feature_names}")
    print(f"Cantidad de variables seleccionadas: {len(feature_names)}")
    print(f"Cantidad de folds temporales: {X.shape[0] - MIN_TRAIN_SAMPLES}")

    print()
    print("Iniciando comparación temporal leave-one-out...")

    results_df = temporal_leave_one_out_comparison(
        X=X,
        y=y,
        metadata_df=metadata_df,
        feature_names=feature_names,
        target_variable=target_variable,
    )

    summary_df = build_metrics_summary(results_df)
    complexity_df = build_model_complexity_summary(results_df)

    with pd.ExcelWriter(OUTPUT_RESULTS_EXCEL, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="resumen_metricas", index=False)
        complexity_df.to_excel(writer, sheet_name="complejidad_modelos", index=False)
        results_df.to_excel(writer, sheet_name="predicciones_por_fold", index=False)

    print()
    print("Comparación finalizada.")
    print()
    print("Resumen de métricas:")
    print(summary_df)

    print()
    print("Resumen de complejidad:")
    print(complexity_df)

    print()
    print(f"Archivo generado: {OUTPUT_RESULTS_EXCEL}")



In [44]:
modelos()

Dispositivo usado: cpu

Dataset cargado y ordenado temporalmente.
X.shape: (36, 24, 7)
y.shape: (36,)
Variable objetivo: vsup
Variables seleccionadas: ['vsup', 'vcc', 'velec', 'vmcame', 'argiva', 'ica_arg', 'bc_arg']
Cantidad de variables seleccionadas: 7
Cantidad de folds temporales: 24

Iniciando comparación temporal leave-one-out...
Fold temporal: entrenar [0:12] -> validar muestra 12
Fold temporal: entrenar [0:13] -> validar muestra 13
Fold temporal: entrenar [0:14] -> validar muestra 14
Fold temporal: entrenar [0:15] -> validar muestra 15
Fold temporal: entrenar [0:16] -> validar muestra 16
Fold temporal: entrenar [0:17] -> validar muestra 17
Fold temporal: entrenar [0:18] -> validar muestra 18
Fold temporal: entrenar [0:19] -> validar muestra 19
Fold temporal: entrenar [0:20] -> validar muestra 20
Fold temporal: entrenar [0:21] -> validar muestra 21
Fold temporal: entrenar [0:22] -> validar muestra 22
Fold temporal: entrenar [0:23] -> validar muestra 23
Fold temporal: entrenar [0

# Comparación Gráfica de predicciones

In [45]:
import pandas as pd
import matplotlib.pyplot as plt


INPUT_RESULTS_EXCEL = "comparacion_modelos_temporal_loo.xlsx"

FORECAST_HORIZON_T = 6
OUTPUT_FORECAST_EXCEL = "pronosticos_T_periodos.xlsx"
OUTPUT_FORECAST_PLOT = "comparacion_pronosticos_T_periodos.png"
OUTPUT_ERRORS_PLOT = "comparacion_errores_T_periodos.png"


def plot_forecasts_T_periods(forecast_df, output_png):
    """
    Grafica valores reales y pronósticos de:
    - SARIMA
    - SARIMAX
    - 1D-CNN
    - TCN
    """

    x = forecast_df["periodo"].astype(str)

    plt.figure(figsize=(13, 6))

    plt.plot(
        x,
        forecast_df["y_real"],
        marker="o",
        linewidth=2.5,
        label="Real",
    )

    plt.plot(
        x,
        forecast_df["pred_sarima"],
        marker="o",
        linestyle="--",
        label="SARIMA",
    )

    plt.plot(
        x,
        forecast_df["pred_sarimax"],
        marker="o",
        linestyle="--",
        label="SARIMAX",
    )

    plt.plot(
        x,
        forecast_df["pred_cnn_1d"],
        marker="o",
        linestyle="--",
        label="1D-CNN pequeña",
    )

    plt.plot(
        x,
        forecast_df["pred_tcn"],
        marker="o",
        linestyle="--",
        label="TCN pequeña",
    )

    plt.title(f"Comparación de pronósticos - últimos {len(forecast_df)} periodos")
    plt.xlabel("Periodo")
    plt.ylabel("Valor")
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(output_png, dpi=300)
    plt.close()


def plot_absolute_errors_T_periods(forecast_df, output_png):
    """
    Grafica el error absoluto de cada modelo por periodo.
    Esto ayuda a ver qué modelo se comporta mejor en cada punto.
    """

    x = forecast_df["periodo"].astype(str)

    plt.figure(figsize=(13, 6))

    plt.plot(
        x,
        forecast_df["abs_error_sarima"],
        marker="o",
        linestyle="-",
        label="SARIMA",
    )

    plt.plot(
        x,
        forecast_df["abs_error_sarimax"],
        marker="o",
        linestyle="-",
        label="SARIMAX",
    )

    plt.plot(
        x,
        forecast_df["abs_error_cnn_1d"],
        marker="o",
        linestyle="-",
        label="1D-CNN pequeña",
    )

    plt.plot(
        x,
        forecast_df["abs_error_tcn"],
        marker="o",
        linestyle="-",
        label="TCN pequeña",
    )

    plt.title(f"Errores absolutos por periodo - últimos {len(forecast_df)} periodos")
    plt.xlabel("Periodo")
    plt.ylabel("Error absoluto")
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    plt.savefig(output_png, dpi=300)
    plt.close()


def comparacion_grafica():
    results_df = pd.read_excel(
        INPUT_RESULTS_EXCEL,
        sheet_name="predicciones_por_fold",
    )

    # Intentamos usar el periodo real desde la metadata.
    # En el script anterior suele quedar como meta_target_period.
    if "meta_target_period" in results_df.columns:
        period_col = "meta_target_period"
    else:
        period_col = "val_position_chronological"

    forecast_df = results_df.tail(FORECAST_HORIZON_T).copy()

    forecast_df = forecast_df.rename(
        columns={
            period_col: "periodo",
        }
    )

    required_columns = [
        "periodo",
        "y_real",
        "pred_sarima",
        "pred_sarimax",
        "pred_cnn_1d",
        "pred_tcn",
        "error_sarima",
        "error_sarimax",
        "error_cnn_1d",
        "error_tcn",
        "abs_error_sarima",
        "abs_error_sarimax",
        "abs_error_cnn_1d",
        "abs_error_tcn",
    ]

    missing_columns = [
        col for col in required_columns
        if col not in forecast_df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Faltan columnas en el archivo de resultados: "
            + ", ".join(missing_columns)
            + "\nVerificá que el script de comparación haya generado resultados "
              "para SARIMA, SARIMAX, 1D-CNN y TCN."
        )

    forecast_df = forecast_df[required_columns]

    forecast_df.to_excel(OUTPUT_FORECAST_EXCEL, index=False)

    plot_forecasts_T_periods(
        forecast_df=forecast_df,
        output_png=OUTPUT_FORECAST_PLOT,
    )

    plot_absolute_errors_T_periods(
        forecast_df=forecast_df,
        output_png=OUTPUT_ERRORS_PLOT,
    )

    print("Comparación gráfica generada.")
    print(f"Archivo Excel: {OUTPUT_FORECAST_EXCEL}")
    print(f"Gráfico pronósticos: {OUTPUT_FORECAST_PLOT}")
    print(f"Gráfico errores absolutos: {OUTPUT_ERRORS_PLOT}")


In [46]:
comparacion_grafica()

Comparación gráfica generada.
Archivo Excel: pronosticos_T_periodos.xlsx
Gráfico pronósticos: comparacion_pronosticos_T_periodos.png
Gráfico errores absolutos: comparacion_errores_T_periodos.png
